## Hybrid Retriever - Combining Dense and Sparse Retriever

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

/home/divyanshu/Desktop/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Step 1: sample document
docs1 = [
    Document(page_content="Langchain is a framework for developing applications powered by language models. It can be used for chatbots, Generative Question-Answering (GQA), summarization, and much more.", metadata={"source": "Langchain"}),
    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM.", metadata={"source": "FAISS"}),
    Document(page_content="BM25 is a ranking function used by search engines to estimate the relevance of documents to a given search query. It is based on the probabilistic retrieval framework and is an extension of the traditional TF-IDF model.", metadata={"source": "BM25"}),
    Document(page_content="The Eiffle Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower.", metadata={"source": "Eiffel Tower"}),
    Document(page_content="Human Beings are the most intelligent species on Earth. They have the ability to think, reason, and communicate in complex ways. Humans have created civilizations, developed technology, and explored the universe.", metadata={"source": "Humans"}),
    Document(page_content="Langchain can be used to build applications that leverage the power of language models. It provides a set of tools and abstractions that make it easier to work with language models and integrate them into applications.", metadata={"source": "Langchain"}),
    Document(page_content="Python and Javascript are most used for building generative AI applications. Python is known for its simplicity and readability, while JavaScript is widely used for web development and has a large ecosystem of libraries and frameworks.", metadata={"source": "Python and Javascript"}),
]
docs2 = [
    # --- KEYWORD-HEAVY DOCUMENTS (Optimised for BM25) ---
    # These contain exact product IDs, strict technical codes, or unique names.
    # Semantic search will fail here because vectors dilute exact alphanumeric strings.
    Document(
        page_content="Error code ERR-404-X89: Critical database connection timeout. Failover pool exhausted at master node.", 
        metadata={"source": "sysops_manual", "type": "keyword_dependent"}
    ),
    Document(
        page_content="Product Specification Sheet for Item SKU-9921-ALPHA. High-tensile titanium bolts designed exclusively for aerospace landing gear chassis models 7B and 7C.", 
        metadata={"source": "inventory_catalog", "type": "keyword_dependent"}
    ),
    Document(
        page_content="Langchain is an orchestration framework. It heavily utilizes abstractions like PromptTemplates, OutputParsers, and LCEL (LangChain Expression Language) to chain sequential model calls.", 
        metadata={"source": "langchain_deep_dive", "type": "keyword_dependent"}
    ),

    # --- CONCEPTUAL-HEAVY DOCUMENTS (Optimised for Semantic Dense Search) ---
    # These describe abstract themes, emotions, or metaphors without using obvious words.
    # BM25 will fail here because the user will search for keywords that don't exist in the text.
    Document(
        page_content="The ultimate tool for building software that can converse, summarize complex ideas, and reason like a human using deep learning patterns.", 
        metadata={"source": "ai_brochure", "type": "semantic_dependent"}
    ),
    Document(
        page_content="A massive architectural marvel constructed of iron latticework dominating the skyline of the capital city of France, drawing millions of global tourists annually.", 
        metadata={"source": "travel_guide", "type": "semantic_dependent"}
    ),
    Document(
        page_content="Living organisms on Earth characterized by advanced cognitive faculties, language generation, self-awareness, and the capacity to build complex planetary societies.", 
        metadata={"source": "anthropology_notes", "type": "semantic_dependent"}
    ),
    Document(
        page_content="An open-source mathematical library created by Meta's AI research team that executes incredibly rapid multi-dimensional matrix comparisons directly inside RAM.", 
        metadata={"source": "vector_compute_paper", "type": "semantic_dependent"}
    )
]
#Step 2: Dense Retriever FAISS + HuggingFace
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
dense_vector_store = FAISS.from_documents(docs2, embeddings)
dense_retriever = dense_vector_store.as_retriever(search_kwargs={"k": 3})  # Top 3 results

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9451.59it/s]


In [14]:
### sparse Retrievr BM25
sparse_retriever = BM25Retriever.from_documents(docs2)
sparse_retriever.k = 2 ## Top 3 results

## Step 4 Combine Dense and Sparse retrievers using EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],  # Adjust the weights as needed
)
ensemble_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f7d15b2fe10>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f7d15ba1310>, k=2)], weights=[0.7, 0.3])

In [28]:
query ="What is ERR-404-X89?"
results = ensemble_retriever.invoke(query)
# how to see the score of each result, if there is a score attribute in the result object, you can access it like this:

for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Source: {result.metadata['source']}")
    if hasattr(result, 'score'):
        print(f"Score: {result.score}")
    print()

Result 1:
Content: Langchain is an orchestration framework. It heavily utilizes abstractions like PromptTemplates, OutputParsers, and LCEL (LangChain Expression Language) to chain sequential model calls.
Source: langchain_deep_dive

Result 2:
Content: Error code ERR-404-X89: Critical database connection timeout. Failover pool exhausted at master node.
Source: sysops_manual

Result 3:
Content: Product Specification Sheet for Item SKU-9921-ALPHA. High-tensile titanium bolts designed exclusively for aerospace landing gear chassis models 7B and 7C.
Source: inventory_catalog

Result 4:
Content: Living organisms on Earth characterized by advanced cognitive faculties, language generation, self-awareness, and the capacity to build complex planetary societies.
Source: anthropology_notes



### RAG PipeLine with Hybrid retriever

In [16]:
from langchain.chat_models.base import init_chat_model
from langchain_classic.prompts import PromptTemplate, ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [19]:
## Step 5: Prompt Template for the LLM
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant that answers questions based on the context provided. If the answer is not contained within the context, please respond with "I don't know."

Context:
{context}

Question:
{input}

Answer:
""")

##  Step 6: Initialize the LLM
llm = init_chat_model(
    model="gpt-4o", 
    model_provider="openai",
    temperature=0.0,
)

In [20]:
## Create stuff chain to combine documents and generate answer
stuff_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
)

## Create full rag chain with retriever and stuff chain
rag_chain = create_retrieval_chain(
    retriever=ensemble_retriever,
    combine_docs_chain=stuff_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f7d15b2fe10>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f7d15ba1310>, k=2)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, tem

In [ ]:
rag_query = "what is ERR-404-X89?"
rag_result = rag_chain.invoke({"input": rag_query})
print(f"Answer: {rag_result['answer']}")
print("\n\nSource Documents:")
for i, doc in enumerate(rag_result['context']):
    print(f"Source Document {i+1}:")
    print(f"Content: {doc.page_content}")
    print(f"Source: {doc.metadata['source']}")
    print()


Answer: I don't know.


Source Documents:
Source Document 1:
Content: Living organisms on Earth characterized by advanced cognitive faculties, language generation, self-awareness, and the capacity to build complex planetary societies.
Source: anthropology_notes

Source Document 2:
Content: Product Specification Sheet for Item SKU-9921-ALPHA. High-tensile titanium bolts designed exclusively for aerospace landing gear chassis models 7B and 7C.
Source: inventory_catalog

Source Document 3:
Content: The ultimate tool for building software that can converse, summarize complex ideas, and reason like a human using deep learning patterns.
Source: ai_brochure

Source Document 4:
Content: An open-source mathematical library created by Meta's AI research team that executes incredibly rapid multi-dimensional matrix comparisons directly inside RAM.
Source: vector_compute_paper

